# Concept vector sparsity: SNMF vs SAE2

Compares zero-fraction statistics of the concept vectors (dictionary atoms) produced by the two decomposition methods run via `scripts/run_full_pipeline_dl.sh` on the ImageNet10 subset (`a_parachute, cassette_player, chainsaw, charch, dog, fish, french_horn, garbage_truck, gas_station, golf_ball`).

Reports, per method:
- **exact_zero_fraction**: literal % of entries == 0.0 (dense decompositions like SNMF/SAE are almost never exactly zero unless a sparsity mechanism forces it).
- **near_zero_fraction (1% relative)**: % of entries below 1% of that atom's own max |value| — scale-invariant, unlike a fixed absolute threshold.
- Concept bank shape and per-atom magnitude stats.

In [1]:
import glob
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Fixed-order categorical colors (colorblind-safe pair), one per method —
# never cycled/reassigned so a method always maps to the same color.
METHOD_COLORS = {"snmf": "#4E79A7", "sae2": "#F28E2B"}

OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "outputs/imagenet10")
if not os.path.isabs(OUTPUT_DIR):
    OUTPUT_DIR = os.path.join(os.getcwd(), "..", OUTPUT_DIR)

METHODS = ["snmf", "sae2"]
CONCEPT_FILES = {}
for m in METHODS:
    matches = glob.glob(os.path.join(OUTPUT_DIR, "concept", m, f"combined_concept_{m}_*_raw.pth"))
    if matches:
        CONCEPT_FILES[m] = matches[0]
CONCEPT_FILES

{'snmf': '/media/NVME_8TB/abka03/Projects/xl-vlms/notebooks/../outputs/imagenet10/concept/snmf/combined_concept_snmf_raw.pth',
 'sae2': '/media/NVME_8TB/abka03/Projects/xl-vlms/notebooks/../outputs/imagenet10/concept/sae2/combined_concept_sae2_raw.pth'}

In [2]:
def orient_atoms(C: torch.Tensor) -> torch.Tensor:
    """Ensure rows are atoms (n_atoms x n_features)."""
    return C if C.size(0) <= C.size(1) else C.T.contiguous()


def exact_zero_fraction(C: torch.Tensor) -> float:
    return (C == 0).float().mean().item()


def near_zero_fraction(C: torch.Tensor, threshold: float = 1e-2) -> float:
    """Fraction of entries below `threshold` of each atom's own max |value|
    (relative, so it's comparable across decomposition methods with very
    different natural scales — an absolute cutoff would not be)."""
    atom_scale = C.abs().max(dim=1, keepdim=True).values.clamp_min(1e-12)
    return (C.abs() < threshold * atom_scale).float().mean().item()


def load_concepts(path: str) -> torch.Tensor:
    blob = torch.load(path, map_location="cpu")
    C = torch.as_tensor(blob["concepts"], dtype=torch.float32)
    return orient_atoms(C)


results = {}
for method, path in CONCEPT_FILES.items():
    if not os.path.exists(path):
        print(f"[{method}] MISSING: {path} — run the pipeline first (see run_full_pipeline_dl.sh).")
        continue
    C = load_concepts(path)
    results[method] = {
        "shape": tuple(C.shape),
        "exact_zero_fraction": exact_zero_fraction(C),
        "near_zero_fraction_1pct": near_zero_fraction(C, threshold=1e-2),
        "mean_abs": C.abs().mean().item(),
        "max_abs": C.abs().max().item(),
    }

for method, stats in results.items():
    print(f"=== {method} ===")
    for k, v in stats.items():
        print(f"  {k}: {v}")
    print()

[snmf] MISSING: /media/NVME_8TB/abka03/Projects/xl-vlms/notebooks/../outputs/imagenet10/concept/snmf/combined_concept_snmf_raw.pth — run the pipeline first (see run_full_pipeline_dl.sh).
[sae2] MISSING: /media/NVME_8TB/abka03/Projects/xl-vlms/notebooks/../outputs/imagenet10/concept/sae2/combined_concept_sae2_raw.pth — run the pipeline first (see run_full_pipeline_dl.sh).


In [3]:
# Bar chart: exact vs near-zero (1%) fraction per method, side by side.
# Form: magnitude comparison across a small fixed set of categories -> bar chart.
if results:
    methods_present = list(results.keys())
    exact_vals = [results[m]["exact_zero_fraction"] * 100 for m in methods_present]
    near_vals = [results[m]["near_zero_fraction_1pct"] * 100 for m in methods_present]

    x = np.arange(len(methods_present))
    width = 0.32

    fig, ax = plt.subplots(figsize=(6, 4.5))
    bars_exact = ax.bar(
        x - width / 2, exact_vals, width,
        label="Exact zero (%)",
        color=[METHOD_COLORS.get(m, "#999999") for m in methods_present],
        alpha=1.0,
    )
    bars_near = ax.bar(
        x + width / 2, near_vals, width,
        label="Near-zero, 1% relative (%)",
        color=[METHOD_COLORS.get(m, "#999999") for m in methods_present],
        alpha=0.5,
    )

    for bars in (bars_exact, bars_near):
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:.1f}", (b.get_x() + b.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=9, color="#333333")

    ax.set_ylabel("% of concept-vector entries")
    ax.set_title("Concept vector sparsity: SNMF vs SAE2")
    ax.set_xticks(x)
    ax.set_xticklabels(methods_present)
    ax.set_ylim(0, 105)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print("No concept banks found yet — run the pipeline first.")

No concept banks found yet — run the pipeline first.
